<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/EWPD4LHC_FULL_PIPELINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EWPD4LHC Full Deterministic Pipeline

Single end-to-end pipeline:
1. Clone repo
2. Run SMEFT linear fit
3. Verify d6lin
4. Build GLS Wilson fit
5. Compute Δχ² and pulls
6. Export results

In [1]:
!git clone https://github.com/ewpd4lhc/ewpd4lhc.git
%cd ewpd4lhc

Cloning into 'ewpd4lhc'...
remote: Enumerating objects: 285, done.
remote: Counting objects: 100% (285/285), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 285 (delta 168), reused 231 (delta 120), pack-reused 0 (from 0)
Receiving objects: 100% (285/285), 447.13 KiB | 9.51 MiB/s, done.
Resolving deltas: 100% (168/168), done.
/content/ewpd4lhc


In [2]:
!python ewpd4lhc.py

Performing SM fit...

Analytic SM fit results:
 Observable   Direct         +-         Fit            +-         Indirect       +-         Pull        
--------------------------------------------------------------------------------------------------------
MW                  80.369        0.013       80.358        0.005       80.356        0.006   -0.9
MZ                 91.1876       0.0021      91.1877       0.0020      91.1970       0.0100    0.0
Gmu [10^-5]     1.16637880   0.00000060   1.16637880   0.00000060   1.16660266   0.00048028    0.0
MH                  125.10         0.11       125.10         0.11       102.82        28.37   -0.0
mt                  172.57         0.58       172.67         0.56       174.19         2.25    0.2
alphas             0.11840      0.00080      0.11863      0.00077      0.12130      0.00271    0.3
Deltaalpha        0.059030     0.000090     0.059022     0.000088     0.058823     0.000463   -0.1
GammaZ              2.4955       0.0023       2.49

In [3]:
from pathlib import Path
yaml_path = Path('ewpd_out.yml')
assert yaml_path.exists(), 'ewpd_out.yml not found.'
txt = yaml_path.read_text()
print('d6lin occurrences:', txt.count('d6lin'))
if txt.count('d6lin') == 0:
    raise RuntimeError('SM-only output — linear SMEFT not active.')

d6lin occurrences: 28


In [4]:
import numpy as np, yaml

operators = ['cHDD','cHQ1','cHQ3','cHWB','cHbq','cHd','cHe11','cHe22','cHe33',
'cHj1','cHj3','cHl111','cHl122','cHl133','cHl311','cHl322','cHl333','cHu','cll1221']
op_to_idx = {op:i for i,op in enumerate(operators)}
N = len(operators)

with open('ewpd_out.yml','r') as f:
    y = yaml.safe_load(f)

obs = list(y.keys())
m = len(obs)

err = np.array([float(y[o]['error']) for o in obs])
Corr = np.eye(m)
for i, oi in enumerate(obs):
    corr_i = y[oi].get('correlation', {}) or {}
    for j, oj in enumerate(obs):
        if oj in corr_i:
            Corr[i,j] = float(corr_i[oj])

Corr = 0.5*(Corr+Corr.T)
Sigma_obs = (err[:,None]*Corr)*err[None,:]

r = np.array([float(y[o]['measurement']) - float(y[o]['smprediction']) for o in obs])

A = np.zeros((m,N))
for i,o in enumerate(obs):
    d6 = y[o].get('d6lin', {}) or {}
    for op,j in op_to_idx.items():
        A[i,j] = float(d6.get(op,0.0))

if np.allclose(A,0):
    raise RuntimeError('Design matrix is zero.')

w, V = np.linalg.eigh(Sigma_obs)
w = np.clip(w,0,None)
w_inv_sqrt = np.array([1/np.sqrt(x) if x>1e-12*w.max() else 0.0 for x in w])
Wm12 = (V*w_inv_sqrt)@V.T

Aw = Wm12@A
rw = Wm12@r

ATA = Aw.T@Aw
ATb = Aw.T@rw

U,s,Vt = np.linalg.svd(ATA,full_matrices=False)
s_inv = np.array([1/x if x>1e-12*s[0] else 0.0 for x in s])
ATA_pinv = (Vt.T*s_inv)@U.T

C_hat = ATA_pinv@ATb
Sigma_C = 0.5*(ATA_pinv+ATA_pinv.T)

print('||C_hat|| =', float(np.linalg.norm(C_hat)))

||C_hat|| = 0.8023146937493355


In [5]:
Sigma_inv = np.linalg.pinv(Sigma_C)
dchi2_sm = float(C_hat.T@Sigma_inv@C_hat)
sig = np.sqrt(np.clip(np.diag(Sigma_C),0,None))
pull = np.divide(C_hat,sig,out=np.zeros_like(C_hat),where=sig>0)

print('Δχ²(SM) =', dchi2_sm)
print('Max |pull| =', float(np.max(np.abs(pull))))

Δχ²(SM) = 11.878821356541167
Max |pull| = 2.9273411902542246


In [7]:
import numpy as np
import pandas as pd

sig = np.sqrt(np.clip(np.diag(Sigma_C), 0, None))
pull = np.divide(C_hat, sig, out=np.zeros_like(C_hat), where=sig>0)

df = pd.DataFrame({"operator": operators, "C_hat": C_hat, "sigma": sig, "pull": pull})
df["abs_pull"] = df["pull"].abs()
df.sort_values("abs_pull", ascending=False).head(10)

,operator,C_hat,sigma,pull,abs_pull
4,cHbq,-0.678004,0.231611,-2.927341,2.927341
1,cHQ1,-0.034866,0.024429,-1.427250,1.427250
2,cHQ3,-0.034866,0.024429,-1.427250,1.427250
6,cHe11,0.061104,0.043022,1.420292,1.420292
0,cHDD,-0.100442,0.081605,-1.230838,1.230838
11,cHl111,0.046297,0.043028,1.075957,1.075957
3,cHWB,0.025272,0.024369,1.037046,1.037046
7,cHe22,0.044943,0.048625,0.924287,0.924287
8,cHe33,0.034833,0.043558,0.799703,0.799703
12,cHl122,0.031285,0.046882,0.667305,0.667305


In [8]:
i = operators.index("cHbq")
C_test = C_hat.copy()
C_test[i] = 0.0

Sigma_inv = np.linalg.pinv(Sigma_C)
dchi2_without = float(C_test.T @ Sigma_inv @ C_test)

print("Δχ² without cHbq =", dchi2_without)

Δχ² without cHbq = 24.90250464070571


In [9]:
col = A[:, operators.index("cHbq")]
idx = np.argsort(np.abs(col))[::-1]

for k in idx[:10]:
    print(obs[k], col[k])

sigmahad 83.25
Re -0.103367
Rmu -0.103367
Rtau -0.103367
Ab 0.0463535
GammaZ -0.0086744
AFBb 0.0074606
Rb -0.00401285
Rc 0.00087891
RWmue 0.0


In [10]:
eigvals, eigvecs = np.linalg.eigh(Sigma_C)
print("Smallest eigenvalue:", eigvals[0])

Smallest eigenvalue: -5.454565696849801e-17


In [11]:
i = operators.index("cHbq")

# profile variance
sigma_prof = 1.0 / np.sqrt(Sigma_inv[i,i])

print("Profile sigma:", sigma_prof)
print("Profile significance:", C_hat[i] / sigma_prof)

Profile sigma: 0.10473234855066893
Profile significance: -6.473681461264472


In [12]:
import numpy as np

i = operators.index("cHbq")

sigma_marg = float(np.sqrt(Sigma_C[i,i]))
pull_marg  = float(C_hat[i] / sigma_marg)
dchi2_prof = float((C_hat[i]**2) / Sigma_C[i,i])

print("cHbq sigma_marg =", sigma_marg)
print("cHbq pull_marg  =", pull_marg)
print("Δχ²_profile(cHbq=0) =", dchi2_prof)
print("equiv sqrt(Δχ²) =", float(np.sqrt(dchi2_prof)))

cHbq sigma_marg = 0.2316108096536138
cHbq pull_marg  = -2.9273411902542246
Δχ²_profile(cHbq=0) = 8.56932644415902
equiv sqrt(Δχ²) = 2.9273411902542246


In [13]:
import numpy as np

i = operators.index("cHbq")

Sigma = Sigma_C
C = C_hat

# Constrained optimum for Gaussian: C' = C - Sigma[:,i] * C_i / Sigma_ii
C_con = C - Sigma[:, i] * (C[i] / Sigma[i,i])
C_con[i] = 0.0

# Check that profiled Δχ² matches
Sinv = np.linalg.pinv(Sigma)
dchi2_full = float(C.T @ Sinv @ C)
dchi2_con  = float(C_con.T @ Sinv @ C_con)
dchi2_prof = dchi2_con - dchi2_full

print("Δχ²(full) =", dchi2_full)
print("Δχ²(constrained best-fit) =", dchi2_con)
print("Δχ²_profile =", dchi2_prof)

# Show the biggest shifts induced by the constraint
delta = C_con - C
idx = np.argsort(np.abs(delta))[::-1]
for k in idx[:10]:
    print(f"{operators[k]:8s}  delta={delta[k]: .6g}   new={C_con[k]: .6g}   old={C[k]: .6g}")

Δχ²(full) = 11.878821356541167
Δχ²(constrained best-fit) = 3.3094949123817425
Δχ²_profile = -8.569326444159424
cHbq      delta= 0.678004   new= 0   old=-0.678004
cHd       delta=-0.464426   new=-0.843052   old=-0.378627
cHu       delta=-0.0740444   new= 0.0280649   old= 0.102109
cHj1      delta=-0.0695551   new=-0.0813382   old=-0.0117831
cHQ3      delta= 0.0521182   new= 0.0172517   old=-0.0348665
cHQ1      delta= 0.0521182   new= 0.0172517   old=-0.0348665
cHDD      delta= 0.029395   new=-0.0710473   old=-0.100442
cHe11     delta=-0.0270184   new= 0.0340858   old= 0.0611042
cHj3      delta=-0.0253939   new=-0.0509426   old=-0.0255487
cHl111    delta=-0.0149636   new= 0.0313331   old= 0.0462967


In [14]:
import numpy as np
import pandas as pd

i = operators.index("cHbq")

# constrained best-fit (conditional mean of others given cHbq=0)
C_con = C_hat - Sigma_C[:, i] * (C_hat[i] / Sigma_C[i,i])
C_con[i] = 0.0

sig = np.sqrt(np.clip(np.diag(Sigma_C), 0, None))
pull_con = np.divide(C_con, sig, out=np.zeros_like(C_con), where=sig>0)

df = pd.DataFrame({
    "operator": operators,
    "C_con": C_con,
    "sigma": sig,
    "pull_con": pull_con,
    "abs_pull_con": np.abs(pull_con),
    "C_hat": C_hat,
})
df.sort_values("abs_pull_con", ascending=False).head(10)

,operator,C_con,sigma,pull_con,abs_pull_con,C_hat
0,cHDD,-0.071047,0.081605,-0.870627,0.870627,-0.100442
6,cHe11,0.034086,0.043022,0.792282,0.792282,0.061104
7,cHe22,0.038220,0.048625,0.786029,0.786029,0.044943
12,cHl122,0.034182,0.046882,0.729104,0.729104,0.031285
5,cHd,-0.843052,1.157463,-0.728362,0.728362,-0.378627
11,cHl111,0.031333,0.043028,0.728196,0.728196,0.046297
10,cHj3,-0.050943,0.070524,-0.722349,0.722349,-0.025549
1,cHQ1,0.017252,0.024429,0.706194,0.706194,-0.034866
2,cHQ3,0.017252,0.024429,0.706194,0.706194,-0.034866
3,cHWB,0.015929,0.024369,0.653635,0.653635,0.025272


In [15]:
import numpy as np

eigvals, eigvecs = np.linalg.eigh(Sigma_C)
i_min = np.argmin(eigvals)

print("Smallest eigenvalue:", eigvals[i_min])
vec = eigvecs[:, i_min]

for op, val in sorted(zip(operators, vec), key=lambda x: -abs(x[1]))[:10]:
    print(op, val)

Smallest eigenvalue: -5.454565696849801e-17
cHDD 0.656871322439845
cHe22 -0.3284368654356738
cHe11 -0.32843676781509173
cHe33 -0.3284367481640626
cHWB -0.2922601610122652
cHu 0.21896033083765748
cHl133 -0.1642221669437543
cHl122 -0.16422083537423887
cHl111 -0.1642207264103352
cHd -0.10950552754578055


In [16]:
import numpy as np

eigvals, eigvecs = np.linalg.eigh(Sigma_C)

# keep only positive modes above numerical threshold
thr = 1e-12 * np.max(np.abs(eigvals))
keep = eigvals > thr

print("Total modes:", len(eigvals))
print("Kept (positive) modes:", int(np.sum(keep)))
print("Dropped ~null modes:", int(np.sum(~keep)))
print("Min kept eig:", float(np.min(eigvals[keep])))

# strongest constrained direction = smallest kept eigenvalue
i_strong = np.where(keep)[0][np.argmin(eigvals[keep])]
v = eigvecs[:, i_strong]

print("\nStrongest constrained direction (smallest positive eigenvalue):")
for op, val in sorted(zip(operators, v), key=lambda x: -abs(x[1]))[:12]:
    print(f"{op:8s} {val:+.6f}")

# weakest constrained among kept = largest kept eigenvalue
i_weak = np.where(keep)[0][np.argmax(eigvals[keep])]
v2 = eigvecs[:, i_weak]

print("\nWeakest constrained direction (largest eigenvalue among kept):")
for op, val in sorted(zip(operators, v2), key=lambda x: -abs(x[1]))[:12]:
    print(f"{op:8s} {val:+.6f}")

Total modes: 19
Kept (positive) modes: 16
Dropped ~null modes: 3
Min kept eig: 4.68025159217897e-06

Strongest constrained direction (smallest positive eigenvalue):
cHWB     +0.834802
cHl322   +0.309093
cHDD     +0.306018
cHl311   +0.249196
cll1221  -0.149934
cHe11    -0.141090
cHj3     -0.073592
cHl111   -0.036263
cHQ1     -0.022956
cHQ3     -0.022956
cHl122   +0.022867
cHe22    -0.018549

Weakest constrained direction (largest eigenvalue among kept):
cHd      +0.986322
cHj1     +0.100143
cHDD     +0.063659
cHu      +0.056990
cHj3     +0.055753
cHe11    -0.031925
cHe33    -0.030868
cHe22    -0.030651
cHbq     -0.027953
cHl333   -0.020859
cHl122   -0.020541
cHl111   -0.019848


In [17]:
import numpy as np

j = operators.index("cHbq")
col = A[:, j]
idx = np.argsort(np.abs(col))[::-1]

print("Top |dO/dcHbq| observables:")
for k in idx[:12]:
    if abs(col[k]) < 1e-12:
        break
    print(f"{obs[k]:10s}  {col[k]: .6g}")

Top |dO/dcHbq| observables:
sigmahad     83.25
Re          -0.103367
Rmu         -0.103367
Rtau        -0.103367
Ab           0.0463535
GammaZ      -0.0086744
AFBb         0.0074606
Rb          -0.00401285
Rc           0.00087891


In [18]:
import numpy as np

j = operators.index("cHbq")
delta_obs = A[:, j] * C_hat[j]

# contribution weighted by inverse covariance of observables
# if you have obs covariance matrix Cov_O:
# contrib = delta_obs * (Cov_O^{-1} @ delta_obs)
# otherwise approximate by squared normalized shifts:

for k in np.argsort(np.abs(delta_obs))[::-1][:10]:
    print(f"{obs[k]:10s}  delta_O = {delta_obs[k]: .6g}")

sigmahad    delta_O = -56.4438
Re          delta_O =  0.0700832
Rmu         delta_O =  0.0700832
Rtau        delta_O =  0.0700832
Ab          delta_O = -0.0314279
GammaZ      delta_O =  0.00588128
AFBb        delta_O = -0.00505832
Rb          delta_O =  0.00272073
Rc          delta_O = -0.000595904
RWmue       delta_O = -0


In [19]:
import numpy as np

# Column for cHbq
j = operators.index("cHbq")
a = A[:, j]                          # shape (m,)
deltaO = a * C_hat[j]                # predicted observable shift from best-fit cHbq

# Whitening from Sigma_obs already exists from earlier cells
w, V = np.linalg.eigh(Sigma_obs)
w = np.clip(w, 0, None)
w_inv_sqrt = np.array([1/np.sqrt(x) if x>1e-12*w.max() else 0.0 for x in w])
Wm12 = (V*w_inv_sqrt) @ V.T          # Sigma^{-1/2}

u = Wm12 @ r                         # whitened SM residuals
d = Wm12 @ deltaO                    # whitened shift due to cHbq

# Per-observable "leverage" (how much the shift aligns with the residual)
# This is the load-bearing diagnostic.
score = u * d                        # elementwise

idx = np.argsort(np.abs(score))[::-1]
for k in idx[:12]:
    print(f"{obs[k]:10s}  u={u[k]: .4f}  d={d[k]: .4f}  u*d={score[k]: .4f}")

AFBb        u=-2.7685  d=-2.9845  u*d= 8.2626
Rmu         u= 0.8208  d= 2.1955  u*d= 1.8020
Rb          u= 0.4700  d= 3.8324  u*d= 1.8011
Re          u= 0.9378  d= 1.4430  u*d= 1.3533
Ab          u=-0.5916  d=-1.5580  u*d= 0.9218
sigmahad    u=-0.2240  d=-1.6927  u*d= 0.3792
Rtau        u= 0.2003  d= 1.6721  u*d= 0.3349
AFBe        u=-0.5011  d= 0.4557  u*d=-0.2283
Deltaalpha  u= 0.5548  d=-0.3619  u*d=-0.2008
AeSLD       u= 1.5317  d=-0.0925  u*d=-0.1417
AtauLEP     u=-1.1028  d=-0.0996  u*d= 0.1098
GammaZ      u= 0.0498  d= 1.7159  u*d= 0.0855


In [20]:
Sinv = np.linalg.pinv(Sigma_obs)
num = float(a.T @ Sinv @ r)
den = float(a.T @ Sinv @ a)
Chat_1d = num/den
sig_1d = np.sqrt(1/den)
print("1D Chat:", Chat_1d, "1D sigma:", sig_1d, "pull:", Chat_1d/sig_1d)

1D Chat: -0.23365221154369567 1D sigma: 0.10473234391379571 pull: -2.230946074653049


In [21]:
import numpy as np

# eigen-decomposition
eigvals, eigvecs = np.linalg.eigh(Sigma_C)

# keep positive modes
thr = 1e-12 * np.max(np.abs(eigvals))
keep = eigvals > thr

Vpos = eigvecs[:, keep]
lam_pos = eigvals[keep]

# Construct metric in constrained space
Ppos = Vpos @ np.diag(1/lam_pos) @ Vpos.T

print("Effective dimension:", len(lam_pos))

Effective dimension: 16


In [22]:
import numpy as np

# =========================
# STEP 1: constrained metric Ppos (rank-16)
# =========================
eigvals, eigvecs = np.linalg.eigh(Sigma_C)
thr = 1e-12 * np.max(np.abs(eigvals))
keep = eigvals > thr

Vpos = eigvecs[:, keep]
lam_pos = eigvals[keep]
Ppos = Vpos @ np.diag(1.0 / lam_pos) @ Vpos.T

print("Effective dimension:", len(lam_pos))

# =========================
# STEP 2: choose a UV ray R (example: pure cHbq direction)
# Change "cHbq" to any operator you want to test.
# =========================
ray_op = "cHbq"
R = np.zeros(len(operators))
R[operators.index(ray_op)] = 1.0

# Normalize the ray in the Ppos metric
norm = float(np.sqrt(R.T @ Ppos @ R))
if norm == 0.0:
    raise RuntimeError("Ray has zero norm in constrained metric (it lies in null space).")
Rhat = R / norm

print("Testing ray:", ray_op, "| metric-norm =", norm)

# =========================
# STEP 3: decompose best-fit into parallel + perpendicular
# =========================
Chat = np.asarray(C_hat).reshape(-1)

alpha = float(Chat.T @ Ppos @ Rhat)     # coordinate along the ray (metric inner product)
C_par = alpha * Rhat
C_perp = Chat - C_par

# =========================
# STEP 4: compute chi2 decomposition
# =========================
chi2_total = float(Chat.T @ Ppos @ Chat)
chi2_par   = float(C_par.T @ Ppos @ C_par)
chi2_perp  = float(C_perp.T @ Ppos @ C_perp)

print("\nχ² decomposition (in constrained 16D subspace):")
print("chi2_total =", chi2_total)
print("chi2_par   =", chi2_par)
print("chi2_perp  =", chi2_perp)
print("check (par+perp) =", chi2_par + chi2_perp)

# Optional: how much of the best-fit is explained by the ray
frac = chi2_par / chi2_total if chi2_total > 0 else np.nan
print("fraction along ray =", frac)

Effective dimension: 16
Testing ray: cHbq | metric-norm = 9.548148340400928

χ² decomposition (in constrained 16D subspace):
chi2_total = 11.878821356542739
chi2_par   = 4.9771203497154906
chi2_perp  = 6.9017010068272855
check (par+perp) = 11.878821356542776
fraction along ray = 0.4189910935039141


In [23]:
import numpy as np

def ray_stats(ray_op):
    eigvals, eigvecs = np.linalg.eigh(Sigma_C)
    thr = 1e-12*np.max(np.abs(eigvals))
    keep = eigvals > thr
    Vpos = eigvecs[:, keep]
    lam = eigvals[keep]
    Ppos = Vpos @ np.diag(1/lam) @ Vpos.T

    R = np.zeros(len(operators))
    R[operators.index(ray_op)] = 1.0
    norm = float(np.sqrt(R.T @ Ppos @ R))
    if norm == 0.0:
        return None
    Rhat = R/norm

    C = np.asarray(C_hat).reshape(-1)
    alpha = float(C.T @ Ppos @ Rhat)
    Cpar = alpha*Rhat
    Cperp = C-Cpar

    chi2_tot = float(C.T @ Ppos @ C)
    chi2_par = float(Cpar.T @ Ppos @ Cpar)
    chi2_perp = float(Cperp.T @ Ppos @ Cperp)
    frac = chi2_par/chi2_tot if chi2_tot>0 else np.nan
    return chi2_tot, chi2_par, chi2_perp, frac

cands = ["cHbq","cHd","cHWB","cHDD","cHQ1","cHQ3","cHe11","cHl322","cHl311","cHj3"]
out = []
for op in cands:
    r = ray_stats(op)
    if r is None:
        continue
    out.append((op,)+r)

out.sort(key=lambda x: -x[4])
for op, chi2_tot, chi2_par, chi2_perp, frac in out:
    print(f"{op:7s}  frac={frac: .3f}   chi2_par={chi2_par: .3f}   chi2_perp={chi2_perp: .3f}")

cHbq     frac= 0.419   chi2_par= 4.977   chi2_perp= 6.902
cHQ1     frac= 0.092   chi2_par= 1.090   chi2_perp= 10.789
cHQ3     frac= 0.092   chi2_par= 1.090   chi2_perp= 10.789
cHd      frac= 0.080   chi2_par= 0.953   chi2_perp= 10.926
cHj3     frac= 0.074   chi2_par= 0.875   chi2_perp= 11.004
cHl311   frac= 0.069   chi2_par= 0.824   chi2_perp= 11.055
cHDD     frac= 0.047   chi2_par= 0.562   chi2_perp= 11.317
cHl322   frac= 0.035   chi2_par= 0.414   chi2_perp= 11.465
cHWB     frac= 0.027   chi2_par= 0.316   chi2_perp= 11.563
cHe11    frac= 0.020   chi2_par= 0.237   chi2_perp= 11.642


In [24]:
import numpy as np

# ========= 1. Build constrained metric =========
eigvals, eigvecs = np.linalg.eigh(Sigma_C)
thr = 1e-12 * np.max(np.abs(eigvals))
keep = eigvals > thr

Vpos = eigvecs[:, keep]
lam_pos = eigvals[keep]
Ppos = Vpos @ np.diag(1.0 / lam_pos) @ Vpos.T

print("Effective dimension:", len(lam_pos))

# ========= 2. Define 2D operator basis =========
ops2 = ["cHbq", "cHQ1"]  # <-- change here if desired

R = np.zeros((len(operators), 2))
for i, op in enumerate(ops2):
    R[operators.index(op), i] = 1.0

# ========= 3. Project best-fit into 2D plane =========
Chat = np.asarray(C_hat).reshape(-1)

# Metric Gram matrix in subspace
G = R.T @ Ppos @ R
G_inv = np.linalg.inv(G)

# Projection operator into the 2D plane
Proj = R @ G_inv @ R.T @ Ppos

C_par = Proj @ Chat
C_perp = Chat - C_par

# ========= 4. Compute χ² decomposition =========
chi2_total = float(Chat.T @ Ppos @ Chat)
chi2_par = float(C_par.T @ Ppos @ C_par)
chi2_perp = float(C_perp.T @ Ppos @ C_perp)

print("\n2D subspace:", ops2)
print("chi2_total =", chi2_total)
print("chi2_parallel (2D) =", chi2_par)
print("chi2_perp =", chi2_perp)
print("fraction explained =", chi2_par / chi2_total)

Effective dimension: 16

2D subspace: ['cHbq', 'cHQ1']
chi2_total = 11.878821356542739
chi2_parallel (2D) = 7.38120064219831
chi2_perp = 4.497620714344417
fraction explained = 0.6213748334663537


In [6]:
import json
np.savez_compressed('wilson_linear_export.npz', coeff_names=np.array(operators,dtype='U'), C_hat=C_hat, Sigma_C=Sigma_C)
with open('wilson_space_report.json','w') as f:
    json.dump({'C_hat':C_hat.tolist(),'pull':pull.tolist(),'dchi2_sm':dchi2_sm}, f, indent=2)
print('Export complete.')

Export complete.
